In [2]:
!pip install pypdf sentence-transformers faiss-cpu transformers
!pip install tf-keras


  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ------------- -------------------------- 3.9/11.6 MB 18.1 MB/s eta 0:00:01
   ---------------------------- ----------- 8.1/11.6 MB 20.1 MB/s eta 0:00:01
   ---------------------------------------  11.5/11.6 MB 20.0 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 18.2 MB/s  0:00:00
   ---------------------------------------- 0.0/564.3 kB ? eta -:--:--
   ---------------------------------------- 564.3/564.3 kB 9.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 12.9 MB/s  0:00:00
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------- ----------------------------- 5.0/18.2 MB 23.2 MB/s eta 0:00:01
   -------------------- ------------------- 9.4/18.2 MB 22.6 MB/s eta 0:00:01
   ----------------------- 

In [3]:
# Step 1: Install dependencies (run once per environment)
import os
os.environ["USE_TF"] = "0"  # Disable TensorFlow/Keras backend


# Step 2: Import libraries

import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Step 3: Read PDF
pdf_path = "input.pdf"  # file in current folder
reader = PdfReader(pdf_path)

# Extract text
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print("Total characters extracted:", len(text))

# Step 4: Chunk text into smaller pieces (for embeddings)
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print(f"Number of chunks: {len(chunks)}")

# Step 5: Create embeddings with SentenceTransformer
#embedder = SentenceTransformer('all-MiniLM-L6-v2')
embedder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

embeddings = embedder.encode(chunks, convert_to_numpy=True)

# Step 6: Store in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Step 7: Load a Question-Answering pipeline
qa_model = pipeline("question-answering", model="deepset/roberta-base-squad2")

# Step 8: Function to query the model
def answer_question(question, top_k=5):
    # Find most similar chunks
    q_embedding = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_embedding, top_k)
    context = " ".join([chunks[i] for i in indices[0]])

    # Get answer from QA model
    result = qa_model(question=question, context=context)
    return result["answer"]

# Example Q&A
print(answer_question("What is the main topic of this document?"))

# Step 9: Save model for deployment
embedder.save("embedding_model")
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
model_name = "deepset/roberta-base-squad2"
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.save_pretrained("qa_model")
tokenizer.save_pretrained("qa_model")

print("Models saved in 'embedding_model' and 'qa_model'.")


Total characters extracted: 3126
Number of chunks: 3


Device set to use cpu


hello java hello world
Models saved in 'embedding_model' and 'qa_model'.


In [4]:
# Ask your question
question = "What is Hashing?"
answer = answer_question(question)

# Print it nicely
print("Q:", question)
print("A:", answer)


Q: What is Hashing?
A: boosts performance
